In [17]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
vp_data_db = "../example_data/vp_data_actors.db"

base_table = "transactions_verbs_wcomps_obl_isik_root_eluskoht_counts"

In [2]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()

In [23]:
query = """SELECT * FROM {tbl}""".format(tbl=base_table)
source3 = pd.read_sql_query(query, con)
source3["elus"] = source3["elus_cnt"]/source3["root_cnt"]
source3["mitte_elus"] = source3["mitte_elus_cnt"]/source3["root_cnt"]

In [24]:
# filter: root_cnt >=1000 & div suhe on 3:1

source4 = source3[source3["root_cnt"]>=1000]
source4 = source4[(source4["elus"]>=0.05) & (source4["mitte_elus"]>=0.05)]
source4 = source4[(source4["elus"]>=3*source4["mitte_elus"]) | (source4["mitte_elus"]>=3*source4["elus"])]

## üks näide

### üldjuht võiks olla elus?

In [26]:
source4[source4["root_word"]=='üldjuht']

,root_word,root_cnt,elus_cnt,mitte_elus_cnt,elus,mitte_elus
407036,üldjuht,3357,182,2071,0.054215,0.61692


In [2]:
con = sqlite3.connect("../vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "../v33.db" AS v33')

In [9]:
query = """
SELECT 
*
from 
transactions_verbs_wcomps_obl_isik
where root_word = 'üldjuht'
"""

source2 = pd.read_sql_query(query, con)
source2

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
0,6602,garanteerima,,11686,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
1,6613,ostma,,11701,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
2,33689,leidma,üles,61565,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
3,44794,töötama,,81431,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
4,50809,valima,,92242,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,29720723,võtma,vastu,53509730,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
3353,29720815,saama,,53509908,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
3354,29852371,huvitama,,53699814,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
3355,29866396,hakkama,vastu,53722204,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi


In [29]:
# mis on mitte kunagi juhud

source2[source2["isik"]=='mitte kunagi']

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
0,6602,garanteerima,,11686,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
1,6613,ostma,,11701,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
2,33689,leidma,üles,61565,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
3,44794,töötama,,81431,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
4,50809,valima,,92242,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,29720723,võtma,vastu,53509730,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
3353,29720815,saama,,53509908,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
3354,29852371,huvitama,,53699814,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi
3355,29866396,hakkama,vastu,53722204,üldjuht,obl,S,"ad,com,sg",UNK,UNK,ad,mitte kunagi,ad,mitte kunagi


In [28]:
# mis on mitte kunagi juhud mis pole ad (kellel/millel)

source2[(source2["isik"]=='mitte kunagi') & (source2["w_case"]!='ad')]

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2


In [15]:
# mis on juhud, kus ei ole otsust?

source2[source2["isik"].isna()]

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
17,149656,leiduma,,271016,üldjuht,obl,S,"ad,com,sg",UNK,UNK,None,None,None,None
18,149872,teadma,,271488,üldjuht,obl,S,"ad,com,sg",UNK,UNK,None,None,None,None
21,170259,juhinduma,,308710,üldjuht,obl,S,"ad,com,sg",UNK,UNK,None,None,None,None
22,172846,piisama,,313534,üldjuht,obl,S,"ad,com,sg",UNK,UNK,None,None,None,None
27,215260,toimima,,390313,üldjuht,obl,S,"com,es,sg",UNK,UNK,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3336,28964787,maksustama,,52362960,üldjuht,obl,S,"ad,com,sg",UNK,UNK,None,None,None,None
3341,29010125,toimuma,,52429986,üldjuht,obl,S,"ad,com,sg",UNK,UNK,None,None,None,None
3346,29031687,olema,,52463107,üldjuht,obl,S,"ad,com,sg",UNK,UNK,None,None,None,None
3347,29169591,viidsima,,52666336,üldjuht,obl,S,"ad,com,sg",UNK,UNK,None,None,None,None


In [19]:
root = "../data_files"
margendused = pd.read_csv(os.path.join(root,"every_verb_case_obl.csv"), sep=";", encoding="utf-8")
margendused

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
0,saama - abl (kellelt/millelt),saama,abl (kellelt/millelt),vahel,vahel,mitte kunagi
1,tulema - abl (kellelt/millelt),tulema,abl (kellelt/millelt),vahel,vahel,mitte kunagi
2,küsima - abl (kellelt/millelt),küsima,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
3,nõudma - abl (kellelt/millelt),nõudma,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
4,võtma - abl (kellelt/millelt),võtma,abl (kellelt/millelt),vahel,vahel,mitte kunagi
...,...,...,...,...,...,...
10574,musitseerima - in (kelles/milles),musitseerima,in (kelles/milles),mitte kunagi,alati,muu
10575,kõigutama - in (kelles/milles),kõigutama,in (kelles/milles),mitte kunagi,mitte kunagi,muu
10576,kätlema - in (kelles/milles),kätlema,in (kelles/milles),mitte kunagi,alati,muu
10577,kõmmutama - in (kelles/milles),kõmmutama,in (kelles/milles),mitte kunagi,alati,mitte kunagi


võtame nt 'leiduma', märgenduste tabelis on leiduma 'ad' -> vahel

In [20]:
margendused[margendused["verb"]=='leiduma']

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
844,leiduma - abl (kellelt/millelt),leiduma,abl (kellelt/millelt),mitte kunagi,alati,muu
2453,leiduma - adit (kellesse/millesse lühike nt kü...,leiduma,adit (kellesse/millesse lühike nt külla/keelde...,mitte kunagi,vahel,muu
2656,leiduma - ad (kellel/millel),leiduma,ad (kellel/millel),vahel,alati,mitte kunagi
4738,leiduma - all (kellele/millele),leiduma,all (kellele/millele),vahel,mitte kunagi,muu
7799,leiduma - el (kellest/millest),leiduma,el (kellest/millest),vahel,vahel,muu
9401,leiduma - ill (kellesse/millesse),leiduma,ill (kellesse/millesse),mitte kunagi,alati,mitte kunagi
9608,leiduma - in (kelles/milles),leiduma,in (kelles/milles),mitte kunagi,alati,mitte kunagi


In [31]:
# palju ad käändes on isik=mitte kunagi ??

margendused[margendused["verb"]=='valima']

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
941,valima - abl (kellelt/millelt),valima,abl (kellelt/millelt),mitte kunagi,vahel,muu
1714,valima - adit (kellesse/millesse lühike nt kül...,valima,adit (kellesse/millesse lühike nt külla/keelde...,vahel,vahel,muu
2739,valima - ad (kellel/millel),valima,ad (kellel/millel),mitte kunagi,alati,muu
4752,valima - all (kellele/millele),valima,all (kellele/millele),vahel,vahel,muu
7695,valima - el (kellest/millest),valima,el (kellest/millest),vahel,vahel,muu
8710,valima - ill (kellesse/millesse),valima,ill (kellesse/millesse),mitte kunagi,vahel,pole kindel
9768,valima - in (kelles/milles),valima,in (kelles/milles),mitte kunagi,alati,muu



### hobune võiks olla elus?

In [32]:
source4[source4["root_word"]=='hobune']

,root_word,root_cnt,elus_cnt,mitte_elus_cnt,elus,mitte_elus
157285,hobune,1589,83,317,0.052234,0.199497


In [33]:
query = """
SELECT 
*
from 
transactions_verbs_wcomps_obl_isik
where root_word = 'hobune'
"""

source2 = pd.read_sql_query(query, con)
source2

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
0,3525,hoolitsema,,6166,hobune,obl,S,"com,gen,pl",UNK,YES,None,None,None,None
1,8681,tulema,,15550,hobune,obl,S,"ad,com,sg",UNK,YES,ad,alati,None,None
2,16026,seostuma,,29582,hobune,obl,S,"com,kom,pl",UNK,YES,None,None,None,None
3,34766,lubama,,63506,hobune,obl,S,"ad,com,sg",UNK,YES,None,None,None,None
4,36313,äritsema,,66295,hobune,obl,S,"com,kom,pl",UNK,YES,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1584,29899137,pakkima,peale,53777102,hobune,obl,S,"all,com,sg",UNK,YES,None,None,None,None
1585,29973650,andma,,53891128,hobune,obl,S,"com,el,sg",UNK,YES,None,None,None,None
1586,30021581,sõitma,,53962459,hobune,obl,S,"com,kom,sg",UNK,YES,None,None,None,None
1587,30021583,tulema,,53962463,hobune,obl,S,"com,kom,sg",UNK,YES,None,None,None,None


In [34]:
# mis on mitte kunagi juhud

source2[source2["isik"]=='mitte kunagi']

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
6,51322,ootama,,93224,hobune,obl,S,"ad,com,sg",UNK,YES,ad,mitte kunagi,ad,mitte kunagi
16,265614,jääma,,478041,hobune,obl,S,"com,el,sg",UNK,YES,el,mitte kunagi,el,mitte kunagi
18,313490,jääma,ilma,563368,hobune,obl,S,"com,el,sg",UNK,YES,el,mitte kunagi,el,mitte kunagi
24,599557,puutuma,,1022428,hobune,obl,S,"com,ill,sg",UNK,YES,ill,mitte kunagi,ill,mitte kunagi
36,708116,vahendama,,1229419,hobune,obl,S,"com,el,sg",UNK,YES,el,mitte kunagi,el,mitte kunagi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,27942660,võtma,välja,50786405,hobune,obl,S,"ad,com,sg",UNK,YES,ad,mitte kunagi,ad,mitte kunagi
1532,27942691,võtma,välja,50786464,hobune,obl,S,"ad,com,sg",UNK,YES,ad,mitte kunagi,ad,mitte kunagi
1535,28205370,kõnelema,,51202248,hobune,obl,S,"ad,com,sg",UNK,YES,ad,mitte kunagi,ad,mitte kunagi
1544,28469421,tegema,,51609490,hobune,obl,S,"com,el,sg",UNK,YES,el,mitte kunagi,el,mitte kunagi


In [37]:
source2[source2["isik"]=='alati']

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
1,8681,tulema,,15550,hobune,obl,S,"ad,com,sg",UNK,YES,ad,alati,None,None
17,305559,tulema,,549481,hobune,obl,S,"ad,com,sg",UNK,YES,ad,alati,None,None
25,615198,virutama,,1052158,hobune,obl,S,"all,com,sg",UNK,YES,all,alati,None,None
27,620670,sõitma,otsa,1062572,hobune,obl,S,"all,com,sg",UNK,YES,all,alati,None,None
40,767592,tungima,kallale,1343819,hobune,obl,S,"all,com,sg",UNK,YES,all,alati,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1364,23736360,meeldima,,43815656,hobune,obl,S,"all,com,sg",UNK,YES,all,alati,None,None
1420,24726343,tekkima,,45507612,hobune,obl,S,"ad,com,sg",UNK,YES,ad,alati,None,None
1438,25434726,paluma,,46704136,hobune,obl,S,"abl,com,sg",UNK,YES,abl,alati,None,None
1523,27485620,meeldima,,50094432,hobune,obl,S,"all,com,sg",UNK,YES,all,alati,None,None


In [35]:
# mis on juhud, kus ei ole otsust?

source2[source2["isik"].isna()]

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
0,3525,hoolitsema,,6166,hobune,obl,S,"com,gen,pl",UNK,YES,None,None,None,None
2,16026,seostuma,,29582,hobune,obl,S,"com,kom,pl",UNK,YES,None,None,None,None
3,34766,lubama,,63506,hobune,obl,S,"ad,com,sg",UNK,YES,None,None,None,None
4,36313,äritsema,,66295,hobune,obl,S,"com,kom,pl",UNK,YES,None,None,None,None
5,36533,kirtsutama,,66715,hobune,obl,S,"com,es,sg",UNK,YES,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1584,29899137,pakkima,peale,53777102,hobune,obl,S,"all,com,sg",UNK,YES,None,None,None,None
1585,29973650,andma,,53891128,hobune,obl,S,"com,el,sg",UNK,YES,None,None,None,None
1586,30021581,sõitma,,53962459,hobune,obl,S,"com,kom,sg",UNK,YES,None,None,None,None
1587,30021583,tulema,,53962463,hobune,obl,S,"com,kom,sg",UNK,YES,None,None,None,None


võtame nt 'hoolitsema', mis on kas vahel või mitte kunagi

In [36]:
margendused[margendused["verb"]=='hoolitsema']

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
478,hoolitsema - abl (kellelt/millelt),hoolitsema,abl (kellelt/millelt),mitte kunagi,mitte kunagi,muu
2134,hoolitsema - adit (kellesse/millesse lühike nt...,hoolitsema,adit (kellesse/millesse lühike nt külla/keelde...,vahel,vahel,muu
3104,hoolitsema - ad (kellel/millel),hoolitsema,ad (kellel/millel),mitte kunagi,vahel,muu
5739,hoolitsema - all (kellele/millele),hoolitsema,all (kellele/millele),vahel,vahel,muu ja pole kindel
9532,hoolitsema - ill (kellesse/millesse),hoolitsema,ill (kellesse/millesse),mitte kunagi,vahel,pole kindel
10153,hoolitsema - in (kelles/milles),hoolitsema,in (kelles/milles),mitte kunagi,alati,muu


In [38]:
con.close()